# AF2 — Introducción al Aprendizaje Máquina
## Modelos de aprendizaje supervisado: predicción de stroke
UANL · FIME · Introducción al Aprendizaje Máquina

**Dataset:** `Dataset_limpio.csv` generado en AF1. Se mantiene la misma codificación y las 18 columnas numéricas.

**Importante:** ejecutar las celdas en orden; las métricas se calculan con datos reales, no se presuponen. La clasificación es exploratoria, no un diagnóstico médico.

In [ ]:
# Librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, balanced_accuracy_score)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
np.random.seed(42)
tf.keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)

## Sección 1. División de datos y escalado
La etiqueta es `stroke` (0 = sin derrame, 1 = con derrame). `stratify=y` mantiene aproximadamente la proporción de clases en ambos conjuntos. El escalador se **ajusta solo con entrenamiento**, evitando fuga de información hacia prueba.

In [ ]:
# En Colab, subir Dataset_limpio.csv al panel Archivos o ejecutar la celda siguiente.
archivo = Path('Dataset_limpio.csv')
if not archivo.exists():
    try:
        from google.colab import files
        print('Selecciona Dataset_limpio.csv (salida de AF1)')
        files.upload()
    except ImportError:
        pass
if not archivo.exists():
    raise FileNotFoundError('Falta Dataset_limpio.csv. Descárgalo del trabajo AF1 y súbelo.')
df = pd.read_csv(archivo)
assert 'stroke' in df.columns, 'No se encontró la columna stroke'
assert df['stroke'].dropna().isin([0,1]).all(), 'stroke debe contener 0 y 1'
X = df.drop(columns=['stroke','id'], errors='ignore').copy()
y = df['stroke'].astype(int).copy()
assert not X.isna().any().any(), 'El dataset limpio no debe tener nulos'
assert all(pd.api.types.is_numeric_dtype(X[c]) for c in X), 'Se esperan variables numéricas'
print('Dimensiones:', X.shape, '| Distribución:', y.value_counts().to_dict())
print('Porcentaje positivo:', round(100*y.mean(), 2), '%')
display(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype('float32')
X_test_scaled = scaler.transform(X_test).astype('float32')
print('Entrenamiento:', len(y_train), y_train.value_counts(normalize=True).round(4).to_dict())
print('Prueba:', len(y_test), y_test.value_counts(normalize=True).round(4).to_dict())
print('Media primeras columnas train escalado:', np.mean(X_train_scaled, axis=0)[:5].round(3))

### Investigación: ¿por qué escalar?
Las variables continuas tienen magnitudes diferentes (por ejemplo, glucosa frente a edad). En redes neuronales, una escala muy desigual puede producir gradientes descompensados y hacer más lenta o inestable la optimización. `StandardScaler` centra y estandariza cada característica usando la media y desviación estándar del conjunto de entrenamiento. A diferencia de los árboles de decisión, las redes neuronales suelen beneficiarse de esta transformación. Las columnas binarias también quedan estandarizadas; es válido, aunque no siempre necesario.

## Sección 2. Arquitectura y activaciones
Se construye una red secuencial con **dos capas ocultas** `Dense(32, relu)` y `Dense(16, relu)`, y una neurona de salida `Dense(1, sigmoid)`. `ReLU` facilita entrenar redes profundas al reducir saturación en la región positiva. `Tanh` puede ser útil, pero se satura en los extremos. `Sigmoid` devuelve una probabilidad estimada entre 0 y 1 para clasificación binaria. `binary_crossentropy` mide la calidad de probabilidades de Bernoulli; MSE suele generar gradientes menos adecuados en esta tarea.

In [ ]:
modelo = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation='relu', name='oculta_1'),
    layers.Dense(16, activation='relu', name='oculta_2'),
    layers.Dense(1, activation='sigmoid', name='salida')
], name='red_stroke_AF2')
modelo.summary()

## Sección 3. Entrenamiento, métricas y optimizadores
Se usa **Adam**, que adapta la tasa de aprendizaje a cada parámetro mediante estimaciones de momentos del gradiente. SGD estándar usa una tasa de aprendizaje global; Adam suele progresar más rápido al inicio, aunque no garantiza mejores resultados finales.

Se reportan **AUC-ROC**, **recall**, **precisión** y **AUC-PR** además de accuracy: con ~5 % de casos positivos, predecir siempre la clase 0 puede dar ~95 % de accuracy sin detectar derrames. `EarlyStopping` vigila la pérdida de validación, detiene el entrenamiento si deja de mejorar y recupera los mejores pesos, reduciendo sobreajuste y tiempo de cómputo.

In [ ]:
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[keras.metrics.BinaryAccuracy(name='accuracy'),
             keras.metrics.AUC(name='auc_roc'),
             keras.metrics.AUC(name='auc_pr', curve='PR'),
             keras.metrics.Recall(name='recall'),
             keras.metrics.Precision(name='precision')])
parada = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=12, restore_best_weights=True, min_delta=0.0001)
historial = modelo.fit(
    X_train_scaled, y_train.to_numpy(),
    validation_split=0.20, epochs=120, batch_size=32,
    callbacks=[parada], verbose=1)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,4))
axs[0].plot(historial.history['loss'], label='Entrenamiento')
axs[0].plot(historial.history['val_loss'], label='Validación')
axs[0].set_title('Pérdida binaria'); axs[0].set_xlabel('Época'); axs[0].legend()
axs[1].plot(historial.history['auc_roc'], label='Entrenamiento')
axs[1].plot(historial.history['val_auc_roc'], label='Validación')
axs[1].set_title('AUC-ROC'); axs[1].set_xlabel('Época'); axs[1].legend()
plt.tight_layout(); plt.savefig('curvas_entrenamiento.png', dpi=160); plt.show()
print('Épocas ejecutadas:', len(historial.history['loss']))

In [ ]:
resultados = modelo.evaluate(X_test_scaled, y_test.to_numpy(), verbose=0, return_dict=True)
prob = modelo.predict(X_test_scaled, verbose=0).ravel()
pred = (prob >= 0.5).astype(int)
print('Métricas en prueba (umbral 0.50):')
print(pd.Series(resultados).round(4).to_string())
print('AUC-PR (average precision):', round(average_precision_score(y_test, prob),4))
print('Balanced accuracy:', round(balanced_accuracy_score(y_test,pred),4))
print(classification_report(y_test,pred, digits=3, zero_division=0))
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['Sin stroke','Con stroke'],ax=ax,cmap='Blues')
plt.tight_layout(); plt.savefig('matriz_confusion.png',dpi=160);plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, prob)
precision, recall, _ = precision_recall_curve(y_test, prob)
fig, axs = plt.subplots(1,2,figsize=(12,4))
axs[0].plot(fpr,tpr,label=f'AUC={roc_auc_score(y_test,prob):.3f}')
axs[0].plot([0,1],[0,1],'--',color='gray');axs[0].set(xlabel='FPR',ylabel='TPR',title='Curva ROC');axs[0].legend()
axs[1].plot(recall,precision,label=f'AP={average_precision_score(y_test,prob):.3f}')
axs[1].axhline(y_test.mean(),linestyle='--',color='gray',label='Prevalencia')
axs[1].set(xlabel='Recall',ylabel='Precisión',title='Curva precisión-recall');axs[1].legend()
plt.tight_layout();plt.savefig('curvas_evaluacion.png',dpi=160);plt.show()

### Interpretación y límites
Registra los resultados que aparecen al ejecutar las celdas y discute el compromiso entre recall y precisión. Un umbral de 0.5 es una referencia inicial, no necesariamente el óptimo. Las métricas sobre el conjunto de prueba deben reservarse para la evaluación final. Este conjunto es observacional, está desbalanceado y no permite afirmar eficacia clínica del modelo.

In [ ]:
# Guardar artefactos para reproducibilidad (opcional)
import joblib
modelo.save('modelo_stroke.keras')
joblib.dump({'scaler': scaler, 'columnas': X.columns.tolist()}, 'preprocesamiento.joblib')
pd.DataFrame([resultados]).to_csv('metricas_prueba.csv',index=False)
print('Archivos exportados: modelo_stroke.keras, preprocesamiento.joblib, metricas_prueba.csv')